# Chronos-2 기반 시계열 이상 탐지 (변수별, multivariate)

**아이디어**: 엑셀 파일의 각 변수(컬럼)를 Chronos-2 모델에 **한꺼번에** 입력해서, 변수들 간의 관계까지 참고하며 "한 스텝 앞" 예측을 굴려갑니다(rolling one-step-ahead). 각 변수마다 실제값이 예측 분포(10~90% 구간) 밖으로 크게 벗어나면 그 시점 · 그 변수를 이상(anomaly)으로 표시합니다.

기존 `chronos-bolt-small`(univariate, 변수 하나만 보고 그 변수를 예측)과 달리, **Chronos-2는 여러 변수를 함께 입력하면 서로의 패턴을 참고해서 예측**합니다 — 예: FEED량은 정상인데 그에 맞춰 같이 움직여야 할 온도가 안 움직이는 경우도 잡을 수 있습니다.

**사용 방법**
1. `CONFIG` 셀에서 `EXCEL_PATH`/`SHEET_NAME`이 맞는지 확인 (현재 2CM 운전 데이터로 설정됨)
2. `QUICK_TEST_ROWS`로 우선 최근 5,000시간만 빠르게 검증 → 결과 확인 후 `None`으로 바꿔 전체 재실행
3. 나머지 셀은 순서대로 실행

**환경 확인 결과**: `amazon/chronos-2` 모델이 이 Mac에서 MPS(Apple GPU)로 정상 로드/추론되는 것 확인했습니다. Apache-2.0 라이선스 오픈소스 모델이라 비용 없이 사용 가능합니다.

In [20]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from chronos import BaseChronosPipeline

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

# macOS 기본 폰트(DejaVu Sans)에는 한글 글리프가 없어서 컬럼명의 한글이 깨져(□) 보이는 문제 방지
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False  # AppleGothic엔 유니코드 마이너스(−) 글리프가 없어서 꺼줌


In [ ]:
# ===================== CONFIG =====================
# 파일: "운전데이터_통합_2CM_3CM_4CM.xlsx" -- 2CM/3CM/4CM 운전 데이터를 세로로 합쳐놓은 통합 파일
# (맨 앞 ID 컬럼으로 어느 설비인지 구분함). 헤더는 1번째 행(기존 개별 시트 파일은 2번째 행이었음).
EXCEL_PATH = "운전데이터_통합_2CM_3CM_4CM.xlsx"
SHEET_NAME = "운전데이터_통합"
HEADER_ROW = 0

ID_COL = "ID"                        # 설비 구분 컬럼 ("2CM" / "3CM" / "4CM")
EQUIPMENT_IDS = ["2CM", "3CM", "4CM"]  # 이 순서대로 루프 돌면서 각각 분석

DATE_COL = "근무일자"                # 날짜 컬럼
HOUR_COL = "근무시간"                # 0~23 시간 컬럼 -> DATE_COL과 합쳐 시간축(timestamp) 생성
EXCLUDE_COLS = ["기타\n품종", "기타\n비고"]   # 범주형/텍스트 컬럼은 이상탐지 대상에서 제외

CONTEXT_LENGTH = 168                # 예측에 사용할 과거 구간 길이 (168시간 = 1주일)
PREDICTION_LENGTH = 1               # 한 번에 몇 스텝(시간) 앞을 예측할지
STRIDE = 1                          # 몇 스텝마다 예측을 수행할지 (1 = 매 시점마다)

# 10~90% 구간을 쓰면, 모델이 완벽히 예측해도 통계적으로 약 20%는 원래 그 구간 밖에 나옵니다
# (하위 10% + 상위 10%). 실제 1차 검증에서도 대부분 변수의 "이상 비율"이 18~24%에 몰려있었는데,
# 이는 설비 이상이 아니라 이 임계값 자체의 통계적 특성이었습니다. 그래서 훨씬 좁게(1~99%) 잡습니다.
ANOMALY_QUANTILE_LOW = 0.01          # 이 분위수보다 낮으면 이상
ANOMALY_QUANTILE_HIGH = 0.99         # 이 분위수보다 높으면 이상
SEVERITY_EPS = 1e-3                  # severity(정규화 오차) 계산 시 0으로 나누는 것 방지용 최소값

# 원본 엑셀을 직접 까보니, 예를 들어 "MILL C/M출구온도 GAS"가 103도 근처에서 안정적이다가
# 딱 한 시간만 1103(=103+1000)으로 튀고 바로 다음 시간에 103으로 돌아오는 식의 패턴이 있었음.
# 이런 "한 시점만 고립되어 튀었다가 바로 원래대로 돌아오는" 건 실제 설비 이상이라기보다
# 센서/로깅 오류일 가능성이 매우 높아서(관성이 있는 물리량이 1시간만에 10배 튀었다 복귀 불가능),
# Chronos에 넣기 전에 미리 찾아서 정리함.
CLEAN_DATA_GLITCHES = True
GLITCH_NEIGHBORS = 3               # 앞뒤로 몇 개씩 비교할지
GLITCH_RATIO = 4.0                  # 이웃 baseline 대비 몇 배 이상(또는 1/배 이하) 벗어나야 글리치 후보로 볼지
GLITCH_MIN_ABS_DEV = 0.5            # 최소 절대 편차 (이보다 작은 차이는 무시)
GLITCH_RATIO_BASELINE_FLOOR = 5.0   # baseline이 이 값보다 작으면 비율 검사 대신 z-score 기준만 사용
                                     # (baseline이 0~1처럼 작으면 비율 테스트가 쉽게 폭발해 오탐 남 -> SKEW 변수에서 실제 발견된 버그)
GLITCH_Z_THRESH = 6.0                # 그 변수의 전체 표준편차 대비 몇 배 이상 벗어나야 글리치로 볼지

# 원본 데이터에 시간 갭이 1,946곳 있음(정확히 8시간짜리가 963개, 최대 2일 7시간 -- 정지된 근무조는
# 행이 아예 없음). row 위치 기준으로 컨텍스트 윈도우를 만들면 "168행=168시간" 가정이 깨지고 갭을
# 건너뛴 게 연속 신호처럼 모델에 들어감 (갭 직후 FEED TOTAL 변화량 중앙값이 평소의 16배로 튐 ->
# 갭 경계가 전부 가짜 이상으로 잡힘). 그래서 1시간 grid로 reindex해서 빠진 시간을 NaN으로 남기고
# (Chronos-2는 NaN을 결측 마스크로 처리하도록 설계됨), 아래 시점들은 이상 판정에서 제외함.
MIN_OBSERVATION_RATE = 0.5           # 이 관측률 미만인 컬럼은 자동 제외 (거의 안 채워진 컬럼/LIMS성 실험값)
DOWNTIME_COL = "POLYCOM\n운전시간"    # 설비 정지 판정용 (0~60분, 0이면 그 시간 아예 정지, 97%가 60)
WARMUP_HOURS = 3                     # 정지/갭이 끝난 직후 안정화 구간 (이 시간 동안은 판정 제외)

# Chronos-2: 여러 변수를 한 번에 입력해서 "변수들 간의 관계"까지 보면서 각 변수를 예측하는 multivariate 모델.
# (기존 chronos-bolt-small은 변수를 하나씩 독립적으로만 예측하는 univariate 모델이었음)
MODEL_ID = "amazon/chronos-2"
BATCH_WINDOWS = 128   # 한 번의 모델 호출에 몇 개의 시간 위치(윈도우)를 넣을지. 각 윈도우에는 변수 전체가 같이 들어감

# 설비 1개 x 41개 변수 전체 x 약 4만 시점 x STRIDE=1을 다 돌리면 M-series MPS 기준 약 1시간 소요됩니다.
# (3개 설비를 다 돌리면 3배). 먼저 빠르게 검증하고 싶으면 아래 값을 사용하세요 (검증 끝나면 None으로).
QUICK_TEST_ROWS = 5000    # 설비별 최근 N시간만 사용 (reindex 이후 기준). 전체 데이터로 돌리려면 None

# device 자동 선택: mps(Apple GPU) > cuda > cpu
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Selected device: {DEVICE}")


In [ ]:
# ===================== 데이터 로드 함수 (1시간 grid로 reindex, 설비별 필터링 지원) =====================
def load_data(path, sheet_name, header_row, date_col, hour_col, exclude_cols, min_observation_rate=None,
              equipment_id=None, id_col=None):
    df = pd.read_excel(path, sheet_name=sheet_name, header=header_row)

    if equipment_id is not None and id_col is not None:
        df = df[df[id_col] == equipment_id].reset_index(drop=True)

    raw_time = pd.to_datetime(df[date_col]) + pd.to_timedelta(df[hour_col], unit="h")
    order = raw_time.argsort()
    raw_time = raw_time.iloc[order].reset_index(drop=True)
    df = df.iloc[order].reset_index(drop=True)
    if raw_time.duplicated().any():
        keep = ~raw_time.duplicated()
        print(f"경고: 중복 타임스탬프 {(~keep).sum()}개 발견 -> 첫 값만 사용")
        df, raw_time = df[keep].reset_index(drop=True), raw_time[keep].reset_index(drop=True)

    drop_cols = [c for c in [date_col, hour_col, id_col, *exclude_cols] if c is not None and c in df.columns]
    df = df.drop(columns=drop_cols)

    # 숫자형 컬럼만 이상탐지 대상으로 사용
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric = [c for c in df.columns if c not in numeric_cols]
    if non_numeric:
        print(f"숫자형이 아니라 제외된 컬럼: {non_numeric}")
    df = df[numeric_cols].copy()

    # 1시간 간격의 완전한 시간축으로 reindex. 빠진 시간은 NaN으로 남김 (ffill/bfill 하지 않음 --
    # Chronos-2가 컨텍스트의 NaN을 결측 마스크로 처리하도록 설계돼 있어서 그대로 넣으면 됨).
    df.index = pd.DatetimeIndex(raw_time)
    full_index = pd.date_range(raw_time.min(), raw_time.max(), freq="h")
    n_before = len(df)
    df = df.reindex(full_index)
    n_missing = len(df) - n_before
    print(f"1시간 grid로 reindex: {n_before}행 -> {len(df)}행 (빠진 시간 {n_missing}개를 NaN으로 채움, "
          f"{n_missing/len(df)*100:.1f}%)")
    print(f"reindex 후 전체 NaN 비율: {df.isna().mean().mean()*100:.1f}%")

    # 관측률이 너무 낮은 컬럼(거의 비어있거나, 몇 시간 간격으로만 찍히는 LIMS성 실험값)은 자동 제외
    if min_observation_rate is not None:
        obs_rate = df.notna().mean()
        low_obs_cols = obs_rate[obs_rate < min_observation_rate].index.tolist()
        if low_obs_cols:
            print(f"관측률 {min_observation_rate*100:.0f}% 미만이라 제외된 컬럼 ({len(low_obs_cols)}개):")
            for c in low_obs_cols:
                print(f"  - {c.strip()}: 관측률 {obs_rate[c]*100:.2f}%")
            df = df.drop(columns=low_obs_cols)

    variable_cols = df.columns.tolist()
    time_index = pd.Series(df.index).reset_index(drop=True)
    data = df.reset_index(drop=True)

    return data, time_index, variable_cols


In [ ]:
# ===================== 데이터 글리치(고립 스파이크) 탐지 함수 =====================
# 한 시점(i)이 "고립된 스파이크"인지 판정:
#  - 앞뒤 이웃들의 median(baseline)과 비교해서 이 점만 크게 벗어나고
#  - 그 이웃들 자신은 baseline과 비슷해야 함 (진짜 지속되는 변화가 아니라 딱 한 점만 튄 것)
# "얼마나 벗어나야 크다고 볼지"는 두 기준 중 하나라도 만족하면 됨:
#  - baseline이 충분히 클 때(>= GLITCH_RATIO_BASELINE_FLOOR): baseline 대비 GLITCH_RATIO배 이상/이하
#  - 그 변수의 전체 표준편차 대비 GLITCH_Z_THRESH배 이상 벗어남
# (주의: baseline이 0~1처럼 아주 작을 때 비율만 보면 사소한 변동도 "무한대 비율"로 잡혀서 오탐 남
#  -> 실제로 POLYCOM ROLLER SKEW에서 0<->1 정상 변동이 전부 글리치로 오탐되는 버그가 있었음. 표준편차
#  기준을 병행해서 해결함.)
# NaN 처리: 중심점이나 이웃에 NaN(시간 갭)이 섞여 있으면 비교 자체가 불가능하므로 건너뜀 (판정 안 함).

def detect_spike_glitches(s, global_std, k=GLITCH_NEIGHBORS, ratio_thresh=GLITCH_RATIO,
                           min_abs_dev=GLITCH_MIN_ABS_DEV, ratio_baseline_floor=GLITCH_RATIO_BASELINE_FLOOR,
                           z_thresh=GLITCH_Z_THRESH):
    vals = s.to_numpy(dtype=np.float64)
    n = len(vals)
    flags = np.zeros(n, dtype=bool)
    cleaned = vals.copy()
    for i in range(k, n - k):
        if np.isnan(vals[i]):
            continue
        neighbors = np.concatenate([vals[i - k:i], vals[i + 1:i + 1 + k]])
        if np.isnan(neighbors).any():
            continue  # 갭에 인접한 구간은 이웃 비교가 불가능하므로 건너뜀
        nb_med = np.median(neighbors)
        center = vals[i]
        dev = abs(center - nb_med)
        neighbor_dev = np.abs(neighbors - nb_med)
        isolated = dev > min_abs_dev and np.all(neighbor_dev < dev * 0.3)
        if not isolated:
            continue

        ratio_flag = False
        if abs(nb_med) >= ratio_baseline_floor:
            ratio = center / nb_med
            ratio_flag = ratio > ratio_thresh or ratio < 1.0 / ratio_thresh
        z_flag = dev > z_thresh * max(global_std, 1e-6)

        if ratio_flag or z_flag:
            flags[i] = True
            cleaned[i] = nb_med
    return flags, cleaned


In [24]:
# ===================== Chronos 모델 로드 =====================
pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print(f"Loaded {MODEL_ID} on {DEVICE}")


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loaded amazon/chronos-2 on mps


In [ ]:
# ===================== 변수별 rolling one-step-ahead 예측 (Chronos-2, multivariate) =====================
# 매 시간 위치마다 "그 시점까지의 전체 변수(41개) 과거 데이터"를 한 번에 넣어서 다음 시점의 전체 변수를 예측합니다.
# Chronos-2는 이때 변수들 간의 관계(예: FEED와 온도가 같이 움직이는 패턴)까지 참고해서 각 변수를 예측합니다.
# 컨텍스트에 NaN(시간 갭)이 섞여 있어도 그대로 넣습니다 (Chronos-2가 결측 마스크로 처리).
# 실제값이 예측 분위수(q_low~q_high) 밖으로 벗어나면 이상치인데, 아래는 이상 "판정" 자체를 제외합니다:
#  - gap   : 타깃 시점 실측값이 NaN
#  - down  : 설비 정지 시점 (is_down)
#  - warmup: 정지/갭 직후 안정화 구간 (is_warmup)
# 제외된 시점도 예측값(pred_median 등)은 남겨두되 excluded_reason에 사유를 표시하고 is_anomaly는 False로 둡니다.
#
# severity(심각도)는 "예측 구간 폭 대비 벗어난 정도"가 아니라, "그 변수의 정상 운전 구간 표준편차 대비
# 얼마나 벗어났는지"로 계산합니다 (일종의 z-score, variable_scale은 위에서 정지/갭/웜업 제외하고 계산함).

quantile_levels = pipeline.quantiles  # 모델이 고정으로 제공하는 분위수 목록 (예: 0.01~0.99, 21개)


def nearest_quantile_index(target):
    return min(range(len(quantile_levels)), key=lambda k: abs(quantile_levels[k] - target))


low_i = nearest_quantile_index(ANOMALY_QUANTILE_LOW)
mid_i = nearest_quantile_index(0.5)
high_i = nearest_quantile_index(ANOMALY_QUANTILE_HIGH)
print(f"사용하는 분위수: low={quantile_levels[low_i]}, median={quantile_levels[mid_i]}, high={quantile_levels[high_i]}")


def rolling_forecast_multivariate(values: np.ndarray, variable_cols: list, variable_scale: np.ndarray,
                                   is_down: np.ndarray, is_warmup: np.ndarray, pipeline_obj=None) -> pd.DataFrame:
    """values: shape (n_timesteps, n_variates), NaN 포함 가능. variable_scale: shape (n_variates,).
    is_down/is_warmup: shape (n_timesteps,) -- 그 시점을 이상 판정에서 제외할지 여부.
    pipeline_obj: 사용할 모델(기본은 위에서 로드한 zero-shot pipeline). 파인튜닝한 모델로 비교할 때 넘겨주면 됨."""
    pipeline_obj = pipeline_obj if pipeline_obj is not None else pipeline
    n = values.shape[0]
    values_t = values.T  # (n_variates, n_timesteps)
    positions = list(range(CONTEXT_LENGTH, n, STRIDE))
    if not positions:
        raise ValueError(
            f"데이터 길이({n})가 CONTEXT_LENGTH({CONTEXT_LENGTH})보다 짧습니다. CONTEXT_LENGTH를 줄여주세요."
        )

    rows = []
    for b in range(0, len(positions), BATCH_WINDOWS):
        batch_pos = positions[b : b + BATCH_WINDOWS]
        batch_inputs = np.stack(
            [values_t[:, i - CONTEXT_LENGTH : i] for i in batch_pos]
        )  # (batch, n_variates, CONTEXT_LENGTH), NaN 포함 가능

        forecasts = pipeline_obj.predict(batch_inputs, prediction_length=PREDICTION_LENGTH)
        # forecasts: list of length batch, each shape (n_variates, n_quantiles, prediction_length)

        for j, i in enumerate(batch_pos):
            fc = forecasts[j].detach().to("cpu").float().numpy()  # (n_variates, n_quantiles, prediction_length)
            for v, col in enumerate(variable_cols):
                actual = float(values[i, v])
                q_low = float(fc[v, low_i, 0])
                q_mid = float(fc[v, mid_i, 0])
                q_high = float(fc[v, high_i, 0])

                if np.isnan(actual):
                    reason = "gap"
                    error = np.nan
                    severity = np.nan
                    is_anomaly = False
                else:
                    if is_down[i]:
                        reason = "down"
                    elif is_warmup[i]:
                        reason = "warmup"
                    else:
                        reason = None
                    error = actual - q_mid
                    severity = abs(error) / max(float(variable_scale[v]), SEVERITY_EPS)
                    is_anomaly = (actual < q_low or actual > q_high) if reason is None else False

                rows.append(
                    dict(
                        idx=i,
                        variable=col,
                        actual=actual,
                        pred_median=q_mid,
                        pred_low=q_low,
                        pred_high=q_high,
                        error=error,
                        is_anomaly=is_anomaly,
                        severity=severity,
                        excluded_reason=reason,
                    )
                )

    return pd.DataFrame(rows)


In [ ]:
# ===================== 예측 정확도 계산 함수 =====================
# gap/down/warmup 시점은 빼고 계산하고, naive baseline("직전 시간값")은 idx 기준으로 진짜 1시간 전인
# 경우만 비교합니다 (안 그러면 시간갭을 사이에 두고 "직전값"을 비교하는 오류가 생김).

def compute_accuracy_summary(results_df):
    rows = []
    for col, sub in results_df.groupby("variable"):
        sub = sub.sort_values("idx").reset_index(drop=True)
        actual = sub["actual"].to_numpy()
        pred = sub["pred_median"].to_numpy()
        scored = sub["excluded_reason"].isna().to_numpy()

        naive_pred = np.roll(actual, 1)
        naive_pred[0] = np.nan
        valid = scored & ~np.isnan(naive_pred)
        n_scored = int(valid.sum())
        if n_scored < 2:
            continue

        err = actual[valid] - pred[valid]
        naive_err = actual[valid] - naive_pred[valid]
        mae = np.mean(np.abs(err))
        naive_mae = np.mean(np.abs(naive_err))

        low = sub["pred_low"].to_numpy()[valid]
        high = sub["pred_high"].to_numpy()[valid]
        coverage = ((actual[valid] >= low) & (actual[valid] <= high)).mean() * 100

        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((actual[valid] - actual[valid].mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        rows.append(dict(
            variable=col, label=col.replace("\n", " "),
            mae=mae, naive_mae=naive_mae,
            mae_vs_naive=mae / naive_mae if naive_mae > 0 else np.nan,
            r2=r2, coverage=coverage, n_scored=n_scored,
        ))
    return pd.DataFrame(rows)


In [ ]:
# ===================== 설비별(2CM/3CM/4CM) 파이프라인 실행 =====================
# EQUIPMENT_IDS에 있는 순서대로 각 설비를 독립적으로: 로드(reindex) -> 글리치 정리 ->
# 정지/갭/웜업 판정 -> Chronos-2 예측 -> 요약/정확도 계산까지 수행하고 결과를 딕셔너리에 저장합니다.
# 모델(pipeline)은 한 번만 로드해서 세 설비 모두 재사용합니다.

equipment_results = {}

for equipment_id in EQUIPMENT_IDS:
    print(f"\n{'='*15} {equipment_id} {'='*15}")

    data, time_index, variable_cols = load_data(
        EXCEL_PATH, SHEET_NAME, HEADER_ROW, DATE_COL, HOUR_COL, EXCLUDE_COLS,
        MIN_OBSERVATION_RATE, equipment_id=equipment_id, id_col=ID_COL,
    )
    if QUICK_TEST_ROWS is not None:
        data = data.tail(QUICK_TEST_ROWS).reset_index(drop=True)
        time_index = time_index.tail(QUICK_TEST_ROWS).reset_index(drop=True)
        print(f"QUICK_TEST_ROWS={QUICK_TEST_ROWS} 적용 -> 최근 {len(data)}시간만 사용")
    print(f"변수 개수: {len(variable_cols)}개 (원본 41개 중 {41-len(variable_cols)}개 자동 제외)")

    # 글리치 정리
    glitch_records = []
    if CLEAN_DATA_GLITCHES:
        global_std = data[variable_cols].std()
        for col in variable_cols:
            flags, cleaned = detect_spike_glitches(data[col], global_std[col])
            if flags.any():
                for i in np.where(flags)[0]:
                    glitch_records.append(dict(
                        variable=col, timestamp=time_index[i],
                        original=data[col].iloc[i], replaced_with=cleaned[i],
                    ))
                data[col] = cleaned
    print(f"글리치 {len(glitch_records)}건 정리")

    # 정지/갭/웜업 판정
    is_gap_row = data.isna().all(axis=1).to_numpy()
    is_down = (data[DOWNTIME_COL] == 0).to_numpy()
    is_down_or_gap = is_down | is_gap_row
    is_warmup = np.zeros(len(data), dtype=bool)
    for i in range(1, len(data)):
        if is_down_or_gap[i - 1] and not is_down_or_gap[i]:
            for w in range(WARMUP_HOURS):
                if i + w < len(data):
                    is_warmup[i + w] = True
    score_eligible_mask = ~is_down_or_gap & ~is_warmup
    variable_scale = data.loc[score_eligible_mask, variable_cols].std().to_numpy()
    print(f"gap={is_gap_row.sum()}, down={(is_down & ~is_gap_row).sum()}, "
          f"warmup={(is_warmup & ~is_down_or_gap).sum()}, 정상={score_eligible_mask.sum()} / 전체 {len(data)}")

    # Chronos-2 rolling forecast
    values = data[variable_cols].to_numpy(dtype=np.float32)
    all_results = rolling_forecast_multivariate(values, variable_cols, variable_scale, is_down, is_warmup)
    all_results["timestamp"] = [time_index[i] for i in all_results["idx"]]
    results = {col: df.reset_index(drop=True) for col, df in all_results.groupby("variable")}

    scored = all_results[all_results["excluded_reason"].isna()]
    print(f"스코어링 대상 {len(scored)}개 중 이상 {scored['is_anomaly'].sum()}개 "
          f"({scored['is_anomaly'].mean()*100:.2f}%)")

    summary = (
        scored.groupby("variable")
        .agg(n_points=("is_anomaly", "size"), n_anomalies=("is_anomaly", "sum"), max_severity=("severity", "max"))
        .assign(anomaly_rate=lambda d: d["n_anomalies"] / d["n_points"])
        .sort_values("n_anomalies", ascending=False)
    )
    accuracy_summary = compute_accuracy_summary(all_results)

    equipment_results[equipment_id] = dict(
        data=data, time_index=time_index, variable_cols=variable_cols,
        is_down=is_down, is_warmup=is_warmup, variable_scale=variable_scale,
        all_results=all_results, results=results, summary=summary, accuracy_summary=accuracy_summary,
    )

print(f"\n모든 설비 처리 완료: {list(equipment_results.keys())}")


In [ ]:
# ===================== 설비별 비교 요약 =====================
compare_rows = []
for eid, r in equipment_results.items():
    scored = r["all_results"][r["all_results"]["excluded_reason"].isna()]
    acc = r["accuracy_summary"]
    compare_rows.append(dict(
        설비=eid,
        변수수=len(r["variable_cols"]),
        스코어링대상=len(scored),
        이상탐지수=int(scored["is_anomaly"].sum()),
        이상비율=scored["is_anomaly"].mean() * 100,
        평균_MAE_naive비=acc["mae_vs_naive"].mean(),
        평균_R2=acc["r2"].mean(),
        평균_coverage=acc["coverage"].mean(),
    ))
equipment_comparison = pd.DataFrame(compare_rows).set_index("설비")
print(equipment_comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(equipment_comparison.index, equipment_comparison["이상비율"], color="tab:blue")
axes[0].set_title("설비별 이상 비율 (%)")
axes[1].bar(equipment_comparison.index, equipment_comparison["평균_MAE_naive비"], color="tab:orange")
axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("설비별 평균 MAE/naive (<1이 좋음)")
plt.tight_layout()
plt.show()

equipment_comparison


In [ ]:
# ===================== 상세 분석할 설비 선택 =====================
# 아래 그래프들(월별 시각화, z-score, 정확도, 저장)은 설비 하나씩 자세히 봅니다.
# CURRENT_EQUIPMENT만 바꾸고 이 셀부터 다시 실행하면 다른 설비로 전환됩니다.
CURRENT_EQUIPMENT = "2CM"

_r = equipment_results[CURRENT_EQUIPMENT]
data, time_index, variable_cols = _r["data"], _r["time_index"], _r["variable_cols"]
is_down, is_warmup, variable_scale = _r["is_down"], _r["is_warmup"], _r["variable_scale"]
all_results, results = _r["all_results"], _r["results"]
summary, accuracy_summary = _r["summary"], _r["accuracy_summary"]

print(f"현재 상세 보기: {CURRENT_EQUIPMENT}")
summary


In [ ]:
# ===================== 변수별 시각화 (실제값 vs 예측 구간, 이상치 표시) =====================
# matplotlib 스타일로 복귀. 전체 기간을 한 그래프에 몰아넣지 않고, 달(month) 단위로 쪼개서
# 한 줄에 3개씩(subplot grid) 보여줌 — 기간이 길어도 한 달 단위로는 패턴이 잘 보임.
import math


def plot_variable(col, max_months=None):
    res = results[col]
    months = res["timestamp"].dt.to_period("M")
    periods = sorted(months.unique())
    if max_months is not None:
        periods = periods[-max_months:]

    ncols = 3
    nrows = math.ceil(len(periods) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows), squeeze=False)

    # 달마다 y축이 들쭉날쭉하면 비교가 어려워서, 전체 기간 기준으로 y축 범위를 통일
    y_all = pd.concat([res["actual"], res["pred_low"], res["pred_high"]])
    y_pad = (y_all.max() - y_all.min()) * 0.08 or 1
    y_lim = (y_all.min() - y_pad, y_all.max() + y_pad)

    for idx, period in enumerate(periods):
        ax = axes[idx // ncols][idx % ncols]
        sub = res[months == period]

        ax.fill_between(sub["timestamp"], sub["pred_low"], sub["pred_high"], color="tab:orange", alpha=0.2)
        ax.plot(sub["timestamp"], sub["pred_median"], color="tab:orange", linewidth=1, label="예측 중앙값")
        ax.plot(sub["timestamp"], sub["actual"], color="tab:blue", linewidth=1, label="실제값")
        anomalies = sub[sub["is_anomaly"]]
        ax.scatter(anomalies["timestamp"], anomalies["actual"], color="red", s=14, zorder=5, label="이상 탐지")

        ax.set_title(str(period), fontsize=10)
        ax.set_ylim(y_lim)
        ax.tick_params(axis="x", labelrotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        if idx == 0:
            ax.legend(loc="upper left", fontsize=7, framealpha=0.9)

    # 남는 빈 subplot 숨기기
    for idx in range(len(periods), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    total_anomalies = res["is_anomaly"].sum()
    fig.suptitle(f"{col} — 실제값 vs Chronos-2 예측 (총 {total_anomalies}개 이상 탐지)", fontsize=13, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()


# 이상치가 많은 변수부터 확인 (원하는 변수명으로 바꿔서 호출해도 됩니다)
for col in summary.index[:5]:
    plot_variable(col)


In [ ]:
# ===================== z-score(severity) 시각화 =====================
# 이상 여부(is_anomaly) 자체는 분위수(1~99%)로 판정하지만, "얼마나 벗어났는지"는
# z-score 성격의 severity(변수 자체 표준편차 대비 벗어난 정도)로 따로 보여줍니다.
# 부호(+/-)를 살려서 위/아래 중 어느 쪽으로 벗어났는지도 같이 보이게 함.
# 빨간 점 = is_anomaly(분위수 기준 실제 이상 판정), 빨간 점선 = ±Z_REFERENCE_LINE 참고선(참고용, 판정 기준 아님)

Z_REFERENCE_LINE = 3.0


def plot_variable_zscore(col, max_months=None, z_ref=Z_REFERENCE_LINE):
    res = results[col].copy()
    res["z"] = res["severity"] * np.sign(res["error"])

    months = res["timestamp"].dt.to_period("M")
    periods = sorted(months.unique())
    if max_months is not None:
        periods = periods[-max_months:]

    ncols = 3
    nrows = math.ceil(len(periods) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 2.6 * nrows), squeeze=False)

    z_max = max(res["z"].abs().max(), z_ref) * 1.1

    for idx, period in enumerate(periods):
        ax = axes[idx // ncols][idx % ncols]
        sub = res[months == period]
        colors = np.where(sub["is_anomaly"], "red", "tab:purple")
        ax.scatter(sub["timestamp"], sub["z"], c=colors, s=6, alpha=0.7)
        ax.axhline(0, color="gray", linewidth=0.7)
        ax.axhline(z_ref, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
        ax.axhline(-z_ref, color="red", linestyle="--", linewidth=0.8, alpha=0.6)

        ax.set_title(str(period), fontsize=10)
        ax.set_ylim(-z_max, z_max)
        ax.tick_params(axis="x", labelrotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)

    for idx in range(len(periods), nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    n_anom = res["is_anomaly"].sum()
    fig.suptitle(f"{col} — z-score (severity, 부호 포함) · 총 {n_anom}개 이상", fontsize=13, y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()


for col in summary.index[:5]:
    plot_variable_zscore(col)


In [ ]:
# ===================== 예측 정확도 시각화 =====================
# 이상탐지 이전에, Chronos-2의 "예측 자체"가 얼마나 정확한지 확인.
# - MAE 비교: naive baseline("다음 시간도 지금이랑 같을 것")보다 나은지
# - R^2: 그 변수의 변동을 얼마나 설명하는지 (0에 가까우면 "그냥 평균 찍는 것"과 big difference 없음)
# - 커버리지: 예측 구간(1~99%)에 실제값이 들어온 비율 (이상적으론 (ANOMALY_QUANTILE_HIGH-ANOMALY_QUANTILE_LOW)*100 %)
# (accuracy_summary는 위 "상세 분석할 설비 선택" 셀에서 CURRENT_EQUIPMENT 기준으로 이미 계산되어 있음)

def plot_accuracy_summary(acc=accuracy_summary):
    target_coverage = (ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100
    acc = acc.dropna(subset=["mae_vs_naive"])  # 변동이 전혀 없는 변수(naive_mae=0)는 비율 계산 불가하므로 제외
    acc_sorted = acc.sort_values("mae_vs_naive")
    n = len(acc_sorted)
    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, n * 0.22)))

    ax = axes[0]
    colors = ["tab:blue" if v < 1 else "tab:red" for v in acc_sorted["mae_vs_naive"]]
    ax.barh(acc_sorted["label"], acc_sorted["mae_vs_naive"], color=colors)
    ax.axvline(1.0, color="black", linewidth=1, linestyle="--")
    ax.set_title("Chronos-2 MAE / naive(직전값) MAE\n(<1: Chronos가 더 나음)", fontsize=10)
    ax.tick_params(axis="y", labelsize=7)
    ax.tick_params(axis="x", labelsize=8)
    ax.invert_yaxis()

    ax = axes[1]
    acc_r2 = acc.set_index("label").loc[acc_sorted["label"], "r2"]
    colors2 = ["tab:green" if v >= 0.5 else ("tab:orange" if v >= 0 else "tab:red") for v in acc_r2]
    ax.barh(range(len(acc_r2)), acc_r2.values, color=colors2)
    ax.set_yticks(range(len(acc_r2)))
    ax.set_yticklabels([])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("R² (설명력, 0.5 이상=초록 / 0~0.5=주황 / 음수=빨강)", fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
    ax.invert_yaxis()

    ax = axes[2]
    acc_cov = acc.set_index("label").loc[acc_sorted["label"], "coverage"]
    ax.barh(range(len(acc_cov)), acc_cov.values, color="tab:purple")
    ax.set_yticks(range(len(acc_cov)))
    ax.set_yticklabels([])
    ax.axvline(target_coverage, color="black", linewidth=1, linestyle="--")
    ax.set_title(f"예측구간 커버리지 %\n(점선={target_coverage:.0f}% 이상적)", fontsize=10)
    ax.tick_params(axis="x", labelsize=8)
    ax.set_xlim(min(90, acc_cov.min() - 2), 100)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.show()

    print(f"전체 평균: MAE/naive={acc['mae_vs_naive'].mean():.3f}, R²={acc['r2'].mean():.3f}, "
          f"coverage={acc['coverage'].mean():.2f}% (목표 {target_coverage:.0f}%)")
    print(f"Chronos가 naive보다 나은 변수: {(acc['mae_vs_naive'] < 1).sum()}/{len(acc)}")


plot_accuracy_summary()


In [ ]:
# ===================== 결과 저장 =====================
OUTPUT_PATH = "chronos_anomaly_results.csv"
cols_to_save = ["variable", "timestamp", "actual", "pred_median", "pred_low", "pred_high", "error", "severity", "is_anomaly", "excluded_reason"]

export_df = all_results[cols_to_save].copy()
export_df["variable"] = export_df["variable"].str.replace("\n", " ")  # 컬럼명 안 줄바꿈 제거 -> 엑셀에서 한 줄로 보이게

# encoding="utf-8-sig": UTF-8에 BOM을 붙여줘서 엑셀이 한글을 자동으로 UTF-8로 인식하게 함
# (BOM 없이 저장하면 엑셀이 다른 인코딩으로 잘못 해석해서 한글이 깨져 보임)
export_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"저장 완료: {OUTPUT_PATH}")

# 이상치만 모아서 보기 (스코어링 대상만 -- excluded_reason이 있는 건 애초에 is_anomaly=False라 안 섞임)
all_results[all_results["is_anomaly"]][cols_to_save].sort_values("severity", ascending=False)


# 파인튜닝 실험 (Chronos-2를 이 설비 데이터로 추가 학습)

앞서 본 것처럼 zero-shot Chronos-2는 naive baseline(직전값 그대로 예측)을 절반 정도의 변수에서만 이깁니다. 이 설비 특유의 패턴을 전혀 학습한 적이 없어서인지 확인하기 위해, **LoRA로 가볍게 파인튜닝**해서 같은 테스트 구간(2025-03~11, 위에서 쓴 것과 동일)에서 zero-shot과 정확도를 비교합니다.

- **학습**: ~2024-12-31 (약 5년치)
- **검증**: 2025-01-01 ~ 2025-02-28
- **테스트**: 2025-03-01 ~ (위에서 이미 분석한 구간과 동일, 파인튜닝엔 전혀 사용하지 않음 — 공정한 비교를 위해)

로컬(MPS)에서 LoRA 기준 1000스텝에 약 10분 정도 걸립니다.

In [ ]:
# ===================== 파인튜닝 CONFIG =====================
FT_TRAIN_END = "2025-01-01"     # 이 날짜 이전까지 학습에 사용
FT_VAL_END = "2025-03-01"       # 학습 종료일 ~ 이 날짜 사이는 검증용 (그 이후=테스트 구간엔 손대지 않음)
FT_STRIDE = 24                   # 학습 윈도우를 몇 시간 간격으로 뽑을지 (1시간마다면 너무 많아서 하루 간격으로)
FT_NUM_STEPS = 1000
FT_BATCH_SIZE = 32
FT_LEARNING_RATE = 1e-5          # LoRA 파인튜닝 시 공식 권장값 (일반 파인튜닝의 기본값 1e-6보다 큼)
FT_MODE = "lora"                 # "lora"(가볍고 빠름, 원본 지식 보존) 또는 "full"(느리지만 더 강하게 학습)
FINETUNED_MODEL_DIR = "chronos2_finetuned_2cm"   # 파인튜닝 결과 저장 폴더 (다음에 재학습 없이 재사용 가능)

print(f"Fine-tune mode={FT_MODE}, steps={FT_NUM_STEPS}, batch={FT_BATCH_SIZE}, lr={FT_LEARNING_RATE}")


In [ ]:
# ===================== 파인튜닝용 전체 히스토리 데이터 준비 =====================
# QUICK_TEST_ROWS로 잘라낸 위쪽의 `data`(테스트 구간)와는 별개로, 학습용 전체 히스토리를 다시 불러옵니다.
# CURRENT_EQUIPMENT(위 "상세 분석할 설비 선택" 셀에서 고른 설비)만 필터링해서 씀.
ft_full_data, ft_full_time_index, _ = load_data(
    EXCEL_PATH, SHEET_NAME, HEADER_ROW, DATE_COL, HOUR_COL, EXCLUDE_COLS, MIN_OBSERVATION_RATE,
    equipment_id=CURRENT_EQUIPMENT, id_col=ID_COL,
)

# 위에서 쓴 것과 동일한 글리치 정리 로직을 전체 히스토리에도 적용
if CLEAN_DATA_GLITCHES:
    ft_global_std = ft_full_data[variable_cols].std()
    for col in variable_cols:
        flags, cleaned = detect_spike_glitches(ft_full_data[col], ft_global_std[col])
        if flags.any():
            ft_full_data[col] = cleaned

train_mask = (ft_full_time_index < FT_TRAIN_END).to_numpy()
val_mask = ((ft_full_time_index >= FT_TRAIN_END) & (ft_full_time_index < FT_VAL_END)).to_numpy()

print(f"학습 구간: {ft_full_time_index[train_mask].min()} ~ {ft_full_time_index[train_mask].max()} ({train_mask.sum()}시간)")
print(f"검증 구간: {ft_full_time_index[val_mask].min()} ~ {ft_full_time_index[val_mask].max()} ({val_mask.sum()}시간)")
print(f"테스트 구간(위에서 이미 분석함, 파인튜닝엔 미사용): {time_index.min()} ~ {time_index.max()}")


def build_windows(values, stride, max_nan_frac=0.3):
    """values: (n_timesteps, n_variates), NaN 포함 가능. NaN 비율이 max_nan_frac을 넘는 윈도우는 학습에서 제외."""
    values_t = values.T  # (n_variates, n_timesteps)
    n = values.shape[0]
    total_len = CONTEXT_LENGTH + PREDICTION_LENGTH
    positions = list(range(total_len, n + 1, stride))
    windows = []
    n_skipped = 0
    for i in positions:
        w = values_t[:, i - total_len:i]
        if np.isnan(w).mean() > max_nan_frac:
            n_skipped += 1
            continue
        windows.append(w)
    print(f"  (NaN 비율 {max_nan_frac*100:.0f}% 초과로 제외된 윈도우: {n_skipped}개)")
    return windows


train_values = ft_full_data[variable_cols].to_numpy(dtype=np.float32)[train_mask]
val_values = ft_full_data[variable_cols].to_numpy(dtype=np.float32)[val_mask]
print("학습 윈도우 구성:")
train_windows = build_windows(train_values, FT_STRIDE)
print("검증 윈도우 구성:")
val_windows = build_windows(val_values, FT_STRIDE)
print(f"학습 윈도우 {len(train_windows)}개, 검증 윈도우 {len(val_windows)}개 (윈도우 shape: {train_windows[0].shape})")


In [ ]:
# ===================== 파인튜닝 실행 =====================
pipeline_ft = pipeline.fit(
    inputs=train_windows,
    prediction_length=PREDICTION_LENGTH,
    validation_inputs=val_windows,
    finetune_mode=FT_MODE,
    learning_rate=FT_LEARNING_RATE,
    num_steps=FT_NUM_STEPS,
    batch_size=FT_BATCH_SIZE,
)
pipeline_ft.save_pretrained(FINETUNED_MODEL_DIR)
print(f"파인튜닝 완료 -> 저장 위치: {FINETUNED_MODEL_DIR}")

# 나중에 커널을 재시작한 뒤 재학습 없이 바로 쓰려면:
# from chronos import Chronos2Pipeline
# pipeline_ft = Chronos2Pipeline.from_pretrained(
#     FINETUNED_MODEL_DIR, device_map=DEVICE, torch_dtype=torch.float32,
#     import_allowlist=["chronos.chronos2.model"],
# )


In [ ]:
# ===================== 파인튜닝 모델로 같은 테스트 구간 재예측 =====================
all_results_ft = rolling_forecast_multivariate(values, variable_cols, variable_scale, is_down, is_warmup, pipeline_obj=pipeline_ft)
all_results_ft["timestamp"] = [time_index[i] for i in all_results_ft["idx"]]

scored_ft = all_results_ft[all_results_ft["excluded_reason"].isna()]
print(f"[파인튜닝] 스코어링 대상 {len(scored_ft)}개 중 이상치 {scored_ft['is_anomaly'].sum()}개 "
      f"({scored_ft['is_anomaly'].mean()*100:.2f}%)")


In [ ]:
# ===================== zero-shot vs 파인튜닝 정확도 비교 =====================
accuracy_summary_ft = compute_accuracy_summary(all_results_ft)

compare = accuracy_summary[["variable", "label", "mae_vs_naive", "r2", "coverage"]].merge(
    accuracy_summary_ft[["variable", "mae_vs_naive", "r2", "coverage"]],
    on="variable", suffixes=("_zeroshot", "_finetuned"),
)

print("=== 전체 평균 비교 (zero-shot -> 파인튜닝) ===")
print(f"MAE/naive  : {compare['mae_vs_naive_zeroshot'].mean():.3f} -> {compare['mae_vs_naive_finetuned'].mean():.3f}  (낮을수록 좋음)")
print(f"R^2        : {compare['r2_zeroshot'].mean():.3f} -> {compare['r2_finetuned'].mean():.3f}  (높을수록 좋음)")
print(f"coverage(%): {compare['coverage_zeroshot'].mean():.2f} -> {compare['coverage_finetuned'].mean():.2f}  (목표 "
      f"{(ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100:.0f}%)")
print(f"\nnaive보다 나은 변수 수: zero-shot {int((compare['mae_vs_naive_zeroshot'] < 1).sum())}/{len(compare)} "
      f"-> 파인튜닝 {int((compare['mae_vs_naive_finetuned'] < 1).sum())}/{len(compare)}")

# 변수별 MAE 비율 나란히 비교 (막대그래프)
comp_sorted = compare.sort_values("mae_vs_naive_finetuned")
n = len(comp_sorted)
fig, ax = plt.subplots(figsize=(10, max(6, n * 0.24)))
y = np.arange(n)
ax.barh(y - 0.2, comp_sorted["mae_vs_naive_zeroshot"], height=0.4, color="tab:gray", label="zero-shot")
ax.barh(y + 0.2, comp_sorted["mae_vs_naive_finetuned"], height=0.4, color="tab:blue", label="파인튜닝")
ax.axvline(1.0, color="black", linewidth=1, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(comp_sorted["label"], fontsize=7)
ax.invert_yaxis()
ax.set_title("MAE / naive 비율 -- zero-shot vs 파인튜닝 (<1 이 좋음)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

compare.sort_values("mae_vs_naive_finetuned")
